# Car insurance PDF preprocessing notebook

## Purpose and scope
This notebook preprocesses **Dutch car insurance policy PDFs** from `car_pdfs/` and outputs auditable datasets for retrieval/summarization.

Scope is intentionally narrow:
- car insurance PDFs only
- preprocessing + provenance only
- no retrieval/RAG/ColBERT sections

## Environment and imports
Install dependencies in Colab if needed, then import runtime libraries.

In [ ]:
# Uncomment in Colab when dependencies are missing.
# !pip -q install pymupdf pandas pyarrow tqdm matplotlib spacy
# !python -m spacy download nl_core_news_sm

import json
import math
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import spacy
    SPACY_AVAILABLE = True
except ImportError:
    SPACY_AVAILABLE = False

## Configuration
Defaults keep car-only scope, overlapping chunks, and strong provenance.

> **Drive note:** if your PDFs are on Google Drive, set `PDF_DIR` to your mounted Drive path (example included below).

In [ ]:
# Optional Colab Drive mount for car_pdfs storage.
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    # Uncomment when running in Colab.
    # drive.mount('/content/drive')
    pass

CONFIG: Dict[str, Any] = {
    "PDF_DIR": "./car_pdfs",  # e.g. /content/drive/MyDrive/Applied ML/AML_Pipeline_files/car_pdfs
    "OUT_DIR": "./outputs_car_preprocessing",
    "SAVE_FULL_PAGE_TEXT": False,
    "DOC_MIN_TOTAL_CHARS": 2000,
    "PAGE_LOW_TEXT_CHARS": 200,
    "MAX_LOW_TEXT_PAGE_FRAC": 0.5,
    "REPEATED_LINE_PAGE_FRAC": 0.5,
    "REPEATED_LINE_MAX_CHARS": 80,
    "BLOCK_MERGE_VERTICAL_GAP": 12.0,
    "BLOCK_MERGE_X_TOL": 20.0,
    "CHUNK_TARGET_CHARS": 1200,
    "CHUNK_MIN_CHARS": 400,
    "CHUNK_OVERLAP_CHARS": 250,
    "ALLOW_CROSS_PAGE_CHUNKS": False,
    "MAX_CHARS_FOR_SPACY": 2000,
    "RANDOM_SEED": 42,
}

np.random.seed(CONFIG["RANDOM_SEED"])

OUT_DIR = Path(CONFIG["OUT_DIR"])
(OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "reports").mkdir(parents=True, exist_ok=True)

## Input discovery and validation
Only discover PDFs in the configured car-PDF directory.

In [ ]:
def get_pdf_paths(pdf_dir: Path) -> List[Path]:
    return sorted([p for p in pdf_dir.glob("*.pdf") if p.is_file()])

PDF_DIR = Path(CONFIG["PDF_DIR"])
if not PDF_DIR.exists() or not PDF_DIR.is_dir():
    raise FileNotFoundError(f"PDF_DIR does not exist or is not a directory: {PDF_DIR}")

pdf_paths = get_pdf_paths(PDF_DIR)
if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in: {PDF_DIR}")

print(f"Found {len(pdf_paths)} car insurance PDFs in {PDF_DIR.resolve()}")

## Core helper functions

In [ ]:
def light_clean_text(text: str) -> str:
    if not text:
        return ""
    t = text.replace("­", "")
    t = re.sub(r"-\s*\n\s*", "", t)
    t = t.replace("", "
")
    t = re.sub(r"[ 	]+", " ", t)
    t = re.sub(r"
{3,}", "

", t)
    return t.strip()


def split_lines_clean(text: str) -> List[str]:
    return [ln.strip() for ln in text.splitlines() if ln.strip()]


def detect_repeated_header_footer_lines(
    page_texts: List[str], min_page_frac: float = 0.5, max_chars: int = 80
) -> List[str]:
    n_pages = len(page_texts)
    if n_pages == 0:
        return []

    line_docfreq: Dict[str, int] = {}
    for text in page_texts:
        for ln in set(split_lines_clean(text)):
            if len(ln) <= max_chars:
                line_docfreq[ln] = line_docfreq.get(ln, 0) + 1

    threshold = max(2, math.ceil(min_page_frac * n_pages))
    return sorted([ln for ln, df in line_docfreq.items() if df >= threshold])


def remove_lines(text: str, remove_set: set) -> str:
    if not remove_set:
        return text
    return "
".join([ln for ln in split_lines_clean(text) if ln not in remove_set]).strip()


def blocks_reading_order(blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return sorted(blocks, key=lambda b: (round(float(b["y0"]), 1), round(float(b["x0"]), 1)))


def merge_blocks(blocks: List[Dict[str, Any]], vertical_gap: float, x_tol: float) -> List[List[Dict[str, Any]]]:
    groups: List[List[Dict[str, Any]]] = []
    current: List[Dict[str, Any]] = []

    for block in blocks:
        if not current:
            current = [block]
            continue

        prev = current[-1]
        close_vertically = (float(block["y0"]) - float(prev["y1"])) <= vertical_gap
        aligned = abs(float(block["x0"]) - float(prev["x0"])) <= x_tol

        if close_vertically and aligned:
            current.append(block)
        else:
            groups.append(current)
            current = [block]

    if current:
        groups.append(current)
    return groups


def classify_layout_type(text: str) -> str:
    lines = split_lines_clean(text)
    first_line = lines[0] if lines else ""
    if re.match(r"^(hoofdstuk|artikel)", first_line, flags=re.IGNORECASE):
        return "heading"
    if re.match(r"^\d+(?:\.\d+)*\s+", first_line):
        return "heading"
    if re.match(r"^(\d+[\.)]|[-•])\s+", first_line):
        return "list_item"
    return "paragraph"


def merge_bbox(blocks: List[Dict[str, Any]]) -> List[float]:
    return [
        float(min(b["x0"] for b in blocks)),
        float(min(b["y0"] for b in blocks)),
        float(max(b["x1"] for b in blocks)),
        float(max(b["y1"] for b in blocks)),
    ]


def heading_depth_and_label(text: str) -> Tuple[Optional[int], str]:
    line = split_lines_clean(text)[0] if split_lines_clean(text) else ""
    m = re.match(r"^(\d+(?:\.\d+){0,5})\.?\s+(.*)$", line)
    if m:
        depth = m.group(1).count(".") + 1
        return depth, line
    return None, line


def update_section_stack(section_stack: List[Tuple[int, str]], unit_text: str, unit_type: str) -> List[Tuple[int, str]]:
    if unit_type != "heading":
        return section_stack

    depth, label = heading_depth_and_label(unit_text)
    if not label:
        return section_stack

    if depth is None:
        depth = min(len(section_stack) + 1, 6)

    updated = [(d, t) for (d, t) in section_stack if d < depth]
    updated.append((depth, label))
    return updated


def section_path_from_stack(section_stack: List[Tuple[int, str]]) -> str:
    return " > ".join([txt for _, txt in section_stack])


def find_best_offset(page_text: str, unit_text: str, start_pos: int) -> Tuple[Optional[int], Optional[int], int]:
    """Best-effort exact matching against cleaned page text; no fabricated offsets."""
    if not page_text or not unit_text:
        return None, None, start_pos

    pos = page_text.find(unit_text, start_pos)
    if pos >= 0:
        return pos, pos + len(unit_text), pos + len(unit_text)

    # fallback: search globally and prefer first match at/after current cursor
    all_pos = []
    s = 0
    while True:
        p = page_text.find(unit_text, s)
        if p < 0:
            break
        all_pos.append(p)
        s = p + 1

    if not all_pos:
        return None, None, start_pos

    after_cursor = [p for p in all_pos if p >= start_pos]
    chosen = after_cursor[0] if after_cursor else all_pos[0]
    return chosen, chosen + len(unit_text), chosen + len(unit_text)

## PDF extraction

In [ ]:
@dataclass
class BlockRecord:
    doc_id: str
    filename: str
    page_no: int
    block_id: str
    x0: float
    y0: float
    x1: float
    y1: float
    text_raw_block: str


def extract_pdf(doc_path: Path, config: Dict[str, Any]) -> Tuple[Dict[str, Any], List[Dict[str, Any]], List[BlockRecord], List[str]]:
    doc_id = doc_path.stem
    pages_records: List[Dict[str, Any]] = []
    block_records: List[BlockRecord] = []
    page_texts_for_repetition: List[str] = []

    try:
        pdf = fitz.open(doc_path)
        page_count = len(pdf)
        total_chars = 0

        for pidx in range(page_count):
            page = pdf[pidx]
            page_no = pidx + 1
            page_text_clean = light_clean_text(page.get_text("text") or "")
            page_texts_for_repetition.append(page_text_clean)

            page_char_count = len(page_text_clean)
            total_chars += page_char_count

            for bidx, b in enumerate(page.get_text("blocks")):
                x0, y0, x1, y1, btext, *_ = b
                btext_clean = light_clean_text(btext or "")
                if not btext_clean:
                    continue
                block_records.append(
                    BlockRecord(
                        doc_id=doc_id,
                        filename=doc_path.name,
                        page_no=page_no,
                        block_id=f"{doc_id}_p{page_no}_b{bidx}",
                        x0=float(x0), y0=float(y0), x1=float(x1), y1=float(y1),
                        text_raw_block=btext_clean,
                    )
                )

            pages_records.append(
                {
                    "doc_id": doc_id,
                    "filename": doc_path.name,
                    "page_no": page_no,
                    "page_char_count": page_char_count,
                    "page_text_raw": page_text_clean,
                }
            )

        low_text_pages = sum(r["page_char_count"] < config["PAGE_LOW_TEXT_CHARS"] for r in pages_records)
        doc_record = {
            "doc_id": doc_id,
            "filename": doc_path.name,
            "page_count": page_count,
            "char_count_total": total_chars,
            "extraction_ok": True,
            "low_text_page_frac": (low_text_pages / page_count) if page_count else 1.0,
            "suspect_scan": ((total_chars < config["DOC_MIN_TOTAL_CHARS"]) or ((low_text_pages / page_count) > config["MAX_LOW_TEXT_PAGE_FRAC"])) if page_count else True,
        }
        return doc_record, pages_records, block_records, page_texts_for_repetition

    except Exception as exc:
        return {
            "doc_id": doc_id,
            "filename": doc_path.name,
            "page_count": 0,
            "char_count_total": 0,
            "extraction_ok": False,
            "low_text_page_frac": 1.0,
            "suspect_scan": True,
            "error": repr(exc),
        }, [], [], []


document_rows: List[Dict[str, Any]] = []
pages_rows: List[Dict[str, Any]] = []
block_rows: List[Dict[str, Any]] = []
header_footer_map: Dict[str, Dict[str, Any]] = {}

for pdf_path in tqdm(pdf_paths, desc="Extracting PDFs"):
    doc_row, page_rows, block_recs, page_texts = extract_pdf(pdf_path, CONFIG)
    document_rows.append(doc_row)
    pages_rows.extend(page_rows)
    block_rows.extend([b.__dict__ for b in block_recs])

    repeated_lines = detect_repeated_header_footer_lines(
        page_texts,
        min_page_frac=CONFIG["REPEATED_LINE_PAGE_FRAC"],
        max_chars=CONFIG["REPEATED_LINE_MAX_CHARS"],
    )
    header_footer_map[doc_row["doc_id"]] = {
        "repeated_lines": repeated_lines,
        "n_repeated_lines": len(repeated_lines),
    }

documents_df = pd.DataFrame(document_rows)
pages_df = pd.DataFrame(pages_rows)
blocks_df = pd.DataFrame(block_rows)

## Header/footer detection and cleaning

In [ ]:
if not pages_df.empty:
    pages_df["suspected_header_footer_lines"] = pages_df["doc_id"].map(
        lambda d: header_footer_map.get(d, {}).get("repeated_lines", [])
    )
    pages_df["page_text_no_hf"] = pages_df.apply(
        lambda r: remove_lines(r["page_text_raw"], set(r["suspected_header_footer_lines"])),
        axis=1,
    )

## Atomic unit construction
Atomic units are persisted for auditability. Each unit keeps exact layout provenance (`block_ids`, `bbox`) and best-effort `char_start`/`char_end` offsets into cleaned page text.

In [ ]:
def build_atomic_units(
    doc_id: str,
    pages_df_doc: pd.DataFrame,
    blocks_df_doc: pd.DataFrame,
    repeated_lines: set,
    config: Dict[str, Any],
) -> List[Dict[str, Any]]:
    units: List[Dict[str, Any]] = []
    section_stack: List[Tuple[int, str]] = []
    unit_order_index = 0

    page_text_map = {
        int(r.page_no): str(r.page_text_no_hf) if isinstance(r.page_text_no_hf, str) else ""
        for r in pages_df_doc.itertuples(index=False)
    }
    page_search_cursor: Dict[int, int] = {p: 0 for p in page_text_map.keys()}

    for page_no in sorted(pages_df_doc["page_no"].unique().tolist()):
        page_blocks = blocks_df_doc[blocks_df_doc["page_no"] == page_no].copy()
        if page_blocks.empty:
            continue

        page_blocks["text_clean_no_hf"] = page_blocks["text_raw_block"].apply(
            lambda t: light_clean_text(remove_lines(t, repeated_lines))
        )
        page_blocks = page_blocks[page_blocks["text_clean_no_hf"].str.len() > 0]
        if page_blocks.empty:
            continue

        ordered_blocks = blocks_reading_order([
            {
                "block_id": row.block_id,
                "x0": row.x0,
                "y0": row.y0,
                "x1": row.x1,
                "y1": row.y1,
                "text": row.text_clean_no_hf,
            }
            for row in page_blocks.itertuples(index=False)
        ])

        merged_groups = merge_blocks(
            ordered_blocks,
            vertical_gap=config["BLOCK_MERGE_VERTICAL_GAP"],
            x_tol=config["BLOCK_MERGE_X_TOL"],
        )

        for group in merged_groups:
            text_raw = light_clean_text("\n".join(b["text"] for b in group))
            if not text_raw:
                continue

            chunk_type_layout = classify_layout_type(text_raw)
            section_stack = update_section_stack(section_stack, text_raw, chunk_type_layout)
            section_path = section_path_from_stack(section_stack)

            page_text = page_text_map.get(int(page_no), "")
            cursor = page_search_cursor.get(int(page_no), 0)
            char_start, char_end, new_cursor = find_best_offset(page_text, text_raw, cursor)
            page_search_cursor[int(page_no)] = new_cursor

            units.append(
                {
                    "unit_id": f"{doc_id}_u{unit_order_index:06d}",
                    "doc_id": doc_id,
                    "page_no": int(page_no),
                    "unit_order_index": int(unit_order_index),
                    "block_ids": [b["block_id"] for b in group],
                    "bbox": merge_bbox(group),
                    "section_path": section_path,
                    "chunk_type_layout": chunk_type_layout,
                    "text_raw": text_raw,
                    "char_start": int(char_start) if char_start is not None else None,
                    "char_end": int(char_end) if char_end is not None else None,
                }
            )
            unit_order_index += 1

    return units


## Overlapping chunk construction
Chunks are sliding windows over atomic units, preserving ordered source-unit provenance.

In [ ]:
def _chunk_layout_type(source_unit_types: List[str]) -> str:
    uniq = sorted(set([t for t in source_unit_types if isinstance(t, str) and t]))
    if not uniq:
        return "unknown"
    return uniq[0] if len(uniq) == 1 else "mixed"


def make_overlapping_chunks(doc_id: str, units: List[Dict[str, Any]], config: Dict[str, Any]) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []
    if not units:
        return chunks

    target_chars = config["CHUNK_TARGET_CHARS"]
    min_chars = config["CHUNK_MIN_CHARS"]
    overlap_chars = config["CHUNK_OVERLAP_CHARS"]
    allow_cross_page = config["ALLOW_CROSS_PAGE_CHUNKS"]

    start = 0
    order_index = 0
    n = len(units)

    while start < n:
        current_units: List[Dict[str, Any]] = []
        current_chars = 0
        start_page = units[start]["page_no"]

        i = start
        while i < n:
            u = units[i]
            if not allow_cross_page and current_units and u["page_no"] != start_page:
                break
            if current_units and current_chars >= target_chars:
                break

            current_units.append(u)
            current_chars += len(u["text_raw"]) + 2
            i += 1

        # If chunk starts with heading and there's a next unit, attach at least one following unit.
        if current_units and current_units[0]["chunk_type_layout"] == "heading" and i < n:
            u = units[i]
            if allow_cross_page or u["page_no"] == start_page:
                current_units.append(u)
                current_chars += len(u["text_raw"]) + 2
                i += 1

        if current_chars < min_chars and i < n:
            while i < n:
                u = units[i]
                if not allow_cross_page and current_units and u["page_no"] != start_page:
                    break
                current_units.append(u)
                current_chars += len(u["text_raw"]) + 2
                i += 1
                if current_chars >= min_chars:
                    break

        if not current_units:
            start += 1
            continue

        source_unit_ids = [u["unit_id"] for u in current_units]
        source_unit_types = [u["chunk_type_layout"] for u in current_units]
        source_pages = sorted(set(int(u["page_no"]) for u in current_units))
        block_ids = [bid for u in current_units for bid in u["block_ids"]]

        x0 = min(u["bbox"][0] for u in current_units)
        y0 = min(u["bbox"][1] for u in current_units)
        x1 = max(u["bbox"][2] for u in current_units)
        y1 = max(u["bbox"][3] for u in current_units)

        section_path = next((u["section_path"] for u in current_units if u["section_path"]), "")
        text_raw = "\n\n".join(u["text_raw"] for u in current_units).strip()

        chunks.append(
            {
                "doc_id": doc_id,
                "chunk_id": f"{doc_id}_c{order_index:05d}",
                "page_start": int(current_units[0]["page_no"]),
                "page_end": int(current_units[-1]["page_no"]),
                "source_pages": source_pages,
                "source_unit_ids": source_unit_ids,
                "source_unit_start_idx": int(current_units[0]["unit_order_index"]),
                "source_unit_end_idx": int(current_units[-1]["unit_order_index"]),
                "source_unit_count": int(len(source_unit_ids)),
                "source_unit_types": source_unit_types,
                "block_ids": block_ids,
                "bbox": [x0, y0, x1, y1],
                "section_path": section_path,
                "chunk_type_layout": _chunk_layout_type(source_unit_types),
                "order_index": int(order_index),
                "text_raw": text_raw,
            }
        )

        order_index += 1
        if i >= n:
            break

        # Move start using overlap in characters over atomic units.
        back_chars = 0
        next_start_offset = len(current_units) - 1
        while next_start_offset > 0 and back_chars < overlap_chars:
            back_chars += len(current_units[next_start_offset]["text_raw"]) + 2
            next_start_offset -= 1
        start += max(1, next_start_offset + 1)

    return chunks


def build_document_chunks_and_units(
    doc_id: str,
    pages_df_doc: pd.DataFrame,
    blocks_df_doc: pd.DataFrame,
    config: Dict[str, Any],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    repeated_lines = set(header_footer_map.get(doc_id, {}).get("repeated_lines", []))
    units = build_atomic_units(doc_id, pages_df_doc, blocks_df_doc, repeated_lines, config)
    chunks = make_overlapping_chunks(doc_id, units, config)
    return units, chunks


atomic_unit_rows: List[Dict[str, Any]] = []
chunk_rows: List[Dict[str, Any]] = []

ok_doc_ids = documents_df.loc[documents_df["extraction_ok"], "doc_id"].tolist() if not documents_df.empty else []
for doc_id in tqdm(ok_doc_ids, desc="Building units/chunks"):
    doc_units, doc_chunks = build_document_chunks_and_units(
        doc_id=doc_id,
        pages_df_doc=pages_df[pages_df["doc_id"] == doc_id],
        blocks_df_doc=blocks_df[blocks_df["doc_id"] == doc_id],
        config=CONFIG,
    )
    atomic_unit_rows.extend(doc_units)
    chunk_rows.extend(doc_chunks)

atomic_units_df = pd.DataFrame(atomic_unit_rows)
chunks_df = pd.DataFrame(chunk_rows)

# Enforce stable schemas even when no rows are produced.
atomic_unit_required_cols = [
    "unit_id", "doc_id", "page_no", "unit_order_index", "block_ids", "bbox",
    "section_path", "chunk_type_layout", "text_raw", "char_start", "char_end",
]
chunk_required_cols = [
    "doc_id", "chunk_id", "page_start", "page_end", "source_pages", "source_unit_ids",
    "source_unit_start_idx", "source_unit_end_idx", "source_unit_count", "source_unit_types",
    "block_ids", "bbox", "section_path", "chunk_type_layout", "order_index", "text_raw",
]

for col in atomic_unit_required_cols:
    if col not in atomic_units_df.columns:
        atomic_units_df[col] = pd.Series(dtype="object")
atomic_units_df = atomic_units_df[atomic_unit_required_cols + [c for c in atomic_units_df.columns if c not in atomic_unit_required_cols]]

for col in chunk_required_cols:
    if col not in chunks_df.columns:
        chunks_df[col] = pd.Series(dtype="object")
chunks_df = chunks_df[chunk_required_cols + [c for c in chunks_df.columns if c not in chunk_required_cols]]

print(f"Atomic units: {len(atomic_units_df)} | Chunks: {len(chunks_df)}")


## Optional enrichments

In [ ]:
NLP = None
if SPACY_AVAILABLE:
    try:
        NLP = spacy.load("nl_core_news_sm", disable=["parser", "ner", "textcat"])
    except Exception as exc:
        warnings.warn(f"spaCy model unavailable; text_lexical disabled: {exc}")


def build_text_lexical(text: str, max_chars: int = 2000) -> str:
    if NLP is None or not isinstance(text, str) or not text.strip():
        return ""
    doc = NLP(text[:max_chars])
    return " ".join(
        token.lemma_.strip().lower()
        for token in doc
        if not (token.is_space or token.is_punct or token.is_stop)
    )

if not chunks_df.empty:
    chunks_df["text_lexical"] = [
        build_text_lexical(t, max_chars=CONFIG["MAX_CHARS_FOR_SPACY"])
        for t in tqdm(chunks_df["text_raw"].fillna(""), desc="Lexical preprocessing")
    ]

    section_summary_df = (
        chunks_df.assign(section_path=chunks_df["section_path"].fillna(""))
        .groupby(["doc_id", "section_path"], dropna=False)
        .agg(n_chunks=("chunk_id", "count"), chars=("text_raw", lambda s: int(s.str.len().sum())))
        .reset_index()
        .sort_values(["doc_id", "n_chunks"], ascending=[True, False])
    )
else:
    section_summary_df = pd.DataFrame(columns=["doc_id", "section_path", "n_chunks", "chars"])

## Output saving
Preserved outputs:
- `documents.csv`
- `pages.csv`
- `chunks.parquet`

Added output:
- `atomic_units.parquet`

In [ ]:
def json_serialize_if_needed(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value

pages_to_save = pages_df.copy()
if not CONFIG["SAVE_FULL_PAGE_TEXT"] and "page_text_raw" in pages_to_save.columns:
    pages_to_save = pages_to_save.drop(columns=["page_text_raw"])

# Keep CSV-compatible nested fields.
for col in ["suspected_header_footer_lines"]:
    if col in pages_to_save.columns:
        pages_to_save[col] = pages_to_save[col].map(json_serialize_if_needed)

chunks_to_save = chunks_df.copy()
for col in [
    "source_pages", "source_unit_ids", "source_unit_types",
    "block_ids", "bbox"
]:
    if col in chunks_to_save.columns:
        chunks_to_save[col] = chunks_to_save[col].map(json_serialize_if_needed)

atomic_units_to_save = atomic_units_df.copy()
for col in ["block_ids", "bbox"]:
    if col in atomic_units_to_save.columns:
        atomic_units_to_save[col] = atomic_units_to_save[col].map(json_serialize_if_needed)

documents_path = OUT_DIR / "documents.csv"
pages_path = OUT_DIR / "pages.csv"
chunks_path = OUT_DIR / "chunks.parquet"
atomic_units_path = OUT_DIR / "atomic_units.parquet"

# Always write canonical outputs (including empty tables with headers/schema where supported).
documents_df.to_csv(documents_path, index=False)
pages_to_save.to_csv(pages_path, index=False)
chunks_to_save.to_parquet(chunks_path, index=False)
atomic_units_to_save.to_parquet(atomic_units_path, index=False)

with open(OUT_DIR / "header_footer.json", "w", encoding="utf-8") as f:
    json.dump(header_footer_map, f, ensure_ascii=False, indent=2)

if not section_summary_df.empty:
    section_summary_df.to_csv(OUT_DIR / "reports" / "cluster_summary.csv", index=False)

print(f"Saved outputs to {OUT_DIR.resolve()}")


## QA / diagnostics
Traceability inspection checks:
1. Preview atomic units.
2. Preview chunks.
3. Show all chunks for one selected document in order.
4. Show overlap between consecutive chunks.
5. Show chunk length statistics.
6. Show sample provenance columns.

In [ ]:
qa_summary = {
    "n_documents": int(len(documents_df)),
    "n_pages": int(len(pages_df)),
    "n_atomic_units": int(len(atomic_units_df)),
    "n_chunks": int(len(chunks_df)),
    "failed_docs": int((~documents_df["extraction_ok"]).sum()) if not documents_df.empty else 0,
    "unit_offsets_available_frac": float(atomic_units_df["char_start"].notna().mean()) if not atomic_units_df.empty else 0.0,
    "median_chunk_chars": float(chunks_df["text_raw"].str.len().median()) if not chunks_df.empty else 0.0,
}
with open(OUT_DIR / "reports" / "qa_summary.json", "w", encoding="utf-8") as f:
    json.dump(qa_summary, f, indent=2)

print("Atomic units preview:")
display_cols_units = [
    "unit_id", "doc_id", "page_no", "unit_order_index", "chunk_type_layout",
    "char_start", "char_end", "section_path", "block_ids"
]
print(atomic_units_df[display_cols_units].head(8) if not atomic_units_df.empty else "No units")

print("
Chunks preview:")
display_cols_chunks = [
    "chunk_id", "doc_id", "order_index", "page_start", "page_end",
    "source_unit_start_idx", "source_unit_end_idx", "source_unit_count",
    "chunk_type_layout", "section_path"
]
print(chunks_df[display_cols_chunks].head(8) if not chunks_df.empty else "No chunks")

if not chunks_df.empty:
    selected_doc = chunks_df.iloc[0]["doc_id"]
    doc_chunks = chunks_df[chunks_df["doc_id"] == selected_doc].sort_values("order_index")
    print(f"
All chunks for doc_id={selected_doc}:")
    print(doc_chunks[["chunk_id", "order_index", "source_unit_ids", "text_raw"]].to_string(index=False, max_colwidth=120))

    print("
Overlap between consecutive chunks (source unit id intersection):")
    rows = []
    for i in range(len(doc_chunks) - 1):
        a = doc_chunks.iloc[i]
        b = doc_chunks.iloc[i + 1]
        s1 = set(a["source_unit_ids"] if isinstance(a["source_unit_ids"], list) else json.loads(a["source_unit_ids"]))
        s2 = set(b["source_unit_ids"] if isinstance(b["source_unit_ids"], list) else json.loads(b["source_unit_ids"]))
        rows.append({
            "chunk_a": a["chunk_id"],
            "chunk_b": b["chunk_id"],
            "overlap_count": len(s1.intersection(s2)),
        })
    print(pd.DataFrame(rows).head(12).to_string(index=False) if rows else "Not enough chunks")

    lens = chunks_df["text_raw"].str.len()
    print("
Chunk length stats:")
    print(lens.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_string())

    print("
Sample provenance fields:")
    prov_cols = ["chunk_id", "source_unit_ids", "block_ids", "bbox", "page_start", "page_end", "section_path"]
    print(chunks_df[prov_cols].head(5).to_string(index=False, max_colwidth=120))

    plt.figure(figsize=(8, 5))
    plt.hist(lens, bins=30)
    plt.title("Chunk length distribution")
    plt.xlabel("Characters")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "plots" / "chunk_char_hist.png", dpi=140)
    plt.close()

qa_summary

## Change summary and limitations
- Restored traceability from chunk → atomic units → layout blocks/pages via `source_unit_ids`, `block_ids`, `bbox`, and page span fields.
- Added persisted `atomic_units.parquet` with deterministic `unit_id` and best-effort offsets (`char_start`, `char_end`).
- Assumption: offset matching uses exact cleaned-text matching in reading order; offsets are left null when matching is not reliable.